# Part 1 - Neural Network Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('customer_churn_nn.csv')
df.head()

## Task 1: Dataset Understanding

In [ ]:
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nStatistical Summary:\n", df.describe())

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='churn', data=df)
plt.title('Target Variable Distribution')
plt.show()

## Task 2: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

## Task 3 & 4: Neural Network Model Building, Training and Evaluation

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(128,64),
        activation='relu',
        learning_rate_init=0.001,
        batch_size=32,
        max_iter=200,
        random_state=42
    ))
])

model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Task 5: Hyperparameter Experimentation

In [ ]:
comparison_df = pd.read_csv('results/model_comparison_table.csv')
comparison_df

## Task 6: Final Reflection

### Role of Weights and Biases
Weights determine the importance of input features while biases help shift activation values to improve learning flexibility.

### Why Activation Functions are Required
Activation functions introduce non-linearity, allowing neural networks to learn complex relationships instead of only linear patterns.

### Effect of Learning Rate
- Too high: model may overshoot optimal values and fail to converge.
- Too low: training becomes very slow and may get stuck.

### Underfitting vs Overfitting
The model achieved better training accuracy than testing accuracy, but the gap was moderate, suggesting slight overfitting but acceptable generalization.
